# Unit 1 — Reference Solution: The Light-Rail Stress Test

> ## ⚠️ Disclaimer — AI-generated reference, provided for learning
>

> **© 2026 Ben Galon. All rights reserved.** Part of the Geo-AI course (The Arena). Provided to enrolled students for course use; not for redistribution.
>
> **This solution notebook was generated by the course's AI assistant** (with
> full access to the unit's research synthesis, the demo notebook, and the
> practice task). It is shared with you **from day one, on purpose**: seeing a
> complete, runnable solution shows that the practice task is *feasible* and
> gives you a concrete reference for the *shape* of a strong
> direct -> interpret -> extend answer.
>
> **Two honest caveats:**
> 1. **It has not been fully tested / Run-All'd end-to-end.** It passed static
>    checks (valid notebook JSON; every code cell compiles) but has not had a
>    verified human Colab run. Treat it as a reference, not a guaranteed-runnable
>    artifact — if a cell breaks, fixing it is good practice.
> 2. **The task is open-ended by design — there is no single correct answer.**
>    This is *one* defensible path (reference city: Tel Aviv). Your own city,
>    centrality choice, and definition of "critical" can be just as correct. Do
>    not copy this; use it to check your reasoning. You are graded on the quality
>    of your own direct -> interpret -> extend loop, not on matching this.

**Stack:** identical to the demo — OSMnx v2 + pyrosm + igraph + folium, no
new heavy dependencies. **Reference city:** Tel Aviv (the task default),
which sits in the *same* Israel-and-Palestine Geofabrik extract the demo
used for Jerusalem, so the build pipeline is reused unchanged.

**What it covers:** the baseline (build before/after, remove a route as
car-routable, choose + justify a centrality, recompute, rank rises/falls,
map, defend) and all three extensions: **(a)** compare two proposals,
**(b)** re-run with a different centrality (the *designed surprise*),
**(c)** wrong-class reasoning (topology omits demand/OD/time -> U3/U4).

## 0. Setup

On **Colab** the cell installs the Unit-1 stack via the shared helper. On a
**local** machine it is a no-op (you ran `uv sync --extra unit-1`). Same
setup contract as the demo.

In [ ]:
# Fetch the shared setup helper and install this unit's pinned requirements.
import os, urllib.request

if not os.path.exists("setup_colab.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/bgalon/geo-graph-learning/main/setup_colab.py",
        "setup_colab.py",
    )

from setup_colab import setup_unit
setup_unit("unit-1")

In [ ]:
# Imports + a quick smoke test (fail fast, assert OSMnx v2.x).
import os, time, math
import numpy as np
import networkx as nx
import osmnx as ox
import igraph as ig
import folium

assert ox.__version__ >= "2", f"need OSMnx v2.x, got {ox.__version__}"
print("osmnx", ox.__version__, "| networkx", nx.__version__,
      "| igraph", ig.__version__, "| folium", folium.__version__)

## 1. Build the "before" graph — Tel Aviv primal

Reuses the demo's *operations* pipeline: download the Geofabrik extract
once (cached), cut to a Tel Aviv bounding box with `pyrosm`, then
**simplify -> project to UTM -> consolidate intersections -> keep the LCC**.
A live `ox.graph_from_bbox` pull is kept as a fallback (fine for this
reference notebook; the student demo stays on the self-contained
Geofabrik + pyrosm path).

In [ ]:
# Central Tel Aviv bounding box (WGS84): [min_lon, min_lat, max_lon, max_lat].
TLV_BBOX = [34.760, 32.055, 34.815, 32.110]
GEOFABRIK_PBF_URL = (
    "https://download.geofabrik.de/asia/israel-and-palestine-latest.osm.pbf"
)
PBF_PATH = "/content/israel-and-palestine.osm.pbf"
if not os.path.isdir("/content"):
    PBF_PATH = "israel-and-palestine.osm.pbf"
CONSOLIDATE_TOLERANCE_M = 10.0


def _download_with_retries(url, dest, attempts=4):
    """Cached download with exponential backoff (handles Geofabrik 504s)."""
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f"using cached file: {dest} ({os.path.getsize(dest)/1e6:.0f} MB)")
        return dest
    import requests
    last = None
    for i in range(1, attempts + 1):
        try:
            print(f"downloading (attempt {i}/{attempts}): {url}")
            with requests.get(url, stream=True, timeout=600) as r:
                r.raise_for_status()
                tmp = dest + ".part"
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1 << 20):
                        f.write(chunk)
                os.replace(tmp, dest)
            print(f"saved {dest} ({os.path.getsize(dest)/1e6:.0f} MB)")
            return dest
        except Exception as exc:  # noqa: BLE001
            last = exc
            wait = 2 ** i
            print(f"  attempt {i} failed ({exc}); retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError(f"download failed after {attempts} attempts: {last}")


def _refine_graph(G):
    """simplify -> project (UTM) -> consolidate -> LCC. Shared by both loaders."""
    if "crs" not in G.graph:
        G.graph["crs"] = "epsg:4326"
    try:
        G = ox.simplification.simplify_graph(G)
    except Exception as exc:  # already-simple graphs raise; fine
        print(f"simplify skipped ({exc})")
    G = ox.projection.project_graph(G)  # to UTM BEFORE consolidating
    try:
        G = ox.simplification.consolidate_intersections(
            G, tolerance=CONSOLIDATE_TOLERANCE_M, rebuild_graph=True,
            dead_ends=False)
    except Exception as exc:
        print(f"consolidate skipped ({exc})")
    largest = max(nx.weakly_connected_components(G), key=len)
    G = G.subgraph(largest).copy()
    print(f"LCC: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    return G


def _load_telaviv_geofabrik():
    from pyrosm import OSM
    _download_with_retries(GEOFABRIK_PBF_URL, PBF_PATH)
    print(f"cutting to Tel Aviv bbox {TLV_BBOX} with pyrosm...")
    osm = OSM(PBF_PATH, bounding_box=TLV_BBOX)
    nodes, edges = osm.get_network(network_type="driving", nodes=True)
    if nodes is None or edges is None or len(edges) == 0:
        raise RuntimeError("pyrosm returned no driving network for the bbox.")
    G = osm.to_graph(nodes, edges, graph_type="networkx", osmnx_compatible=True)
    return _refine_graph(G)


def _load_telaviv_live():
    # OSMnx v2: bbox = (left, bottom, right, top) = (W, S, E, N).
    w, s, e, n = TLV_BBOX
    print("FALLBACK: live OSMnx pull (graph_from_bbox)...")
    G = ox.graph_from_bbox(bbox=(w, s, e, n), network_type="drive")
    return _refine_graph(G)


G = None
for _name, _fn in [("Geofabrik+pyrosm", _load_telaviv_geofabrik),
                   ("OSMnx live", _load_telaviv_live)]:
    try:
        print(f"--- source: {_name} ---")
        G = _fn()
        print(f"\u2713 built Tel Aviv primal from: {_name}")
        break
    except Exception as exc:  # noqa: BLE001
        print(f"\u2717 {_name} failed: {exc}")
if G is None:
    raise RuntimeError("Tel Aviv graph build failed (all sources exhausted).")
print(G)

## 2. Centrality helper (igraph-backed, length-weighted)

Same construction as the demo: collapse the LCC to a simple, undirected,
**length-weighted** `Graph`, then compute degree (NetworkX) + closeness and
betweenness (igraph, C-backed, same exact values in seconds). Wrapped in a
function so we can call it on the before *and* after graphs.

In [ ]:
def to_simple_weighted(G_multi):
    """MultiDiGraph -> simple undirected length-weighted Graph (LCC)."""
    largest = max(nx.weakly_connected_components(G_multi), key=len)
    H = G_multi.subgraph(largest).copy()
    GG = nx.Graph()
    for u, v, data in H.edges(data=True):
        w = float(data.get("length", 1.0))
        if (not GG.has_edge(u, v)) or w < GG[u][v]["length"]:
            GG.add_edge(u, v, length=w)
    return GG


def compute_centralities(GG):
    """Return {'degree':..., 'closeness':..., 'betweenness':...} keyed by node id.

    Closeness + betweenness via igraph; betweenness rescaled to match
    nx.betweenness_centrality(..., normalized=True). Length-weighted."""
    deg = dict(GG.degree())
    node_ids = list(GG.nodes())
    idx_of = {nid: i for i, nid in enumerate(node_ids)}
    edges_ig = [(idx_of[u], idx_of[v]) for u, v in GG.edges()]
    weights = [float(GG[u][v]["length"]) for u, v in GG.edges()]
    g = ig.Graph(n=len(node_ids), edges=edges_ig, directed=False)
    g.es["length"] = weights
    clo_vals = g.closeness(weights="length", normalized=True)
    clo = {node_ids[i]: clo_vals[i] for i in range(len(node_ids))}
    btw_raw = g.betweenness(weights="length", directed=False)
    n = g.vcount()
    scale = (2.0 / ((n - 1) * (n - 2))) if n > 2 else 1.0
    btw = {node_ids[i]: btw_raw[i] * scale for i in range(len(node_ids))}
    return {"degree": deg, "closeness": clo, "betweenness": btw}


GG_before = to_simple_weighted(G)
cent_before = compute_centralities(GG_before)
print(f"before: {GG_before.number_of_nodes()} nodes; "
      f"computed degree/closeness/betweenness")

## 3. Baseline — remove a light-rail route, recompute, rank

**DIRECT.** *"On the simplified, UTM-projected, LCC primal graph of central
Tel Aviv, compute length-weighted betweenness; then remove the edges of
<route> as car-routable, recompute on the new LCC, and return the streets
whose betweenness changed most."*

**Why betweenness:** it counts shortest paths passing through a node, so it
is the natural structural proxy for *"carries through-traffic"* — the
planner's actual question (Freeman 1977). It is a structural **hypothesis**,
not measured flow (see extension (c)).

We pick the route by street **name** (Tel Aviv's Allenby St. as the
reference corridor; swap the name for any city). Removing a corridor can
disconnect a fringe -> we **recompute on the new LCC** and state that
decision explicitly.

In [ ]:
def route_edges_by_name(G_multi, names):
    """All (u,v,k) edges whose 'name' matches any string in `names`."""
    want = {names} if isinstance(names, str) else set(names)
    out = []
    for u, v, k, d in G_multi.edges(keys=True, data=True):
        nm = d.get("name")
        nm = nm[0] if isinstance(nm, list) and nm else nm
        if isinstance(nm, str) and nm in want:
            out.append((u, v, k))
    return out


def remove_route(G_multi, route_edges):
    """Drop the route edges, then keep the largest connected component."""
    H = G_multi.copy()
    H.remove_edges_from(route_edges)
    largest = max(nx.weakly_connected_components(H), key=len)
    # One defensible choice: recompute on the NEW largest connected component
    # (removing a street can strand a fringe). In YOUR run, make this call
    # yourself and record it in your decision log — both ways are defensible.
    return H.subgraph(largest).copy()


ROUTE_NAME = "Allenby Street"  # reference corridor; change per city
edges = route_edges_by_name(G, ROUTE_NAME)
print(f"route '{ROUTE_NAME}': {len(edges)} primal edges matched")
if not edges:  # name not found -> fall back to the top-betweenness corridor
    top_node = max(cent_before["betweenness"], key=cent_before["betweenness"].get)
    edges = list(G.edges(top_node, keys=True))
    print(f"  (name not found; removing edges at the top-betweenness node instead)")

G_after = remove_route(G, edges)
GG_after = to_simple_weighted(G_after)
cent_after = compute_centralities(GG_after)
print(f"after: {GG_after.number_of_nodes()} nodes (was {GG_before.number_of_nodes()})")

In [ ]:
# Rank rises and falls in betweenness, joined by node id (NOT row index).
def delta_table(before, after, metric="betweenness"):
    b, a = before[metric], after[metric]
    shared = set(b) & set(a)  # nodes present in both LCCs
    rows = [(nid, a[nid] - b[nid], b[nid], a[nid]) for nid in shared]
    rows.sort(key=lambda r: r[1])
    return rows

rows = delta_table(cent_before, cent_after, "betweenness")
print("TOP 5 RISES (streets that become MORE critical):")
for nid, d, b, a in rows[-5:][::-1]:
    print(f"  node {nid}: +{d:.5f}  ({b:.5f} -> {a:.5f})")
print("\nTOP 5 FALLS (streets that become LESS critical):")
for nid, d, b, a in rows[:5]:
    print(f"  node {nid}: {d:.5f}  ({b:.5f} -> {a:.5f})")

**INTERPRET.** The biggest *rises* are the intersections that absorb the
displaced shortest paths — typically the nearest parallel through-corridor
to the removed route. The *falls* are nodes that were central only because
the removed corridor routed paths through them. **Defence (one paragraph):**
*"Removing Allenby concentrates displaced through-routes onto the streets
around the top-rise nodes; the city should worry about whichever of those
has no parallel relief street nearby. This is a structural prediction from
betweenness, not measured flow."* The honesty sentence is what moves the
paragraph from good-enough to strong.

In [ ]:
# Map the rises (red) and falls (blue) on the AFTER graph over folium.
import folium
rise_nodes = {nid for nid, d, *_ in rows[-15:] if d > 0}
fall_nodes = {nid for nid, d, *_ in rows[:15] if d < 0}

nodes_gdf, edges_gdf = ox.convert.graph_to_gdfs(G_after)
nodes_gdf = nodes_gdf.to_crs(4326)
edges_gdf = edges_gdf.to_crs(4326)
center = [nodes_gdf.geometry.y.mean(), nodes_gdf.geometry.x.mean()]

m = folium.Map(location=center, zoom_start=14, tiles="OpenStreetMap",
               control_scale=True)
folium.GeoJson(edges_gdf[["geometry"]],
               style_function=lambda _f: {"color": "#bbbbbb", "weight": 1,
                                          "opacity": 0.6},
               name="network").add_to(m)
for nid in rise_nodes:
    if nid in nodes_gdf.index:
        row = nodes_gdf.loc[nid]
        folium.CircleMarker([row.geometry.y, row.geometry.x], radius=5,
                            color="#e31a1c", fill=True, fill_opacity=0.85,
                            tooltip="more critical").add_to(m)
for nid in fall_nodes:
    if nid in nodes_gdf.index:
        row = nodes_gdf.loc[nid]
        folium.CircleMarker([row.geometry.y, row.geometry.x], radius=5,
                            color="#1f78b4", fill=True, fill_opacity=0.85,
                            tooltip="less critical").add_to(m)
m

## 4. Extension (a) — compare two proposals

**Define "better" BEFORE comparing** — the definition *is* the modeling
choice (the CHOOSE skill). Here: *better = lower peak replacement load* (the
single most-loaded street after removal is less central -> load is spread,
not piled on one street). A strong answer notices when two definitions
disagree and says which one the planner's question implies.

In [ ]:
def peak_after(route_names):
    """Remove a named route, recompute, return the max betweenness after."""
    e = route_edges_by_name(G, route_names)
    if not e:
        return None
    Ha = remove_route(G, e)
    ca = compute_centralities(to_simple_weighted(Ha))
    return max(ca["betweenness"].values())

PROPOSALS = {
    "A: Allenby Street": "Allenby Street",
    "B: Ibn Gabirol Street": "Ibn Gabirol Street",
}
for label, name in PROPOSALS.items():
    pk = peak_after(name)
    print(f"{label}: peak betweenness after removal = "
          f"{('%.5f' % pk) if pk is not None else 'route not found'}")
print("\nLower peak load => load is more spread => 'better' under this definition.")

## 5. Extension (b) — re-run with a different centrality  *(the designed surprise)*

Re-running the baseline with **closeness** instead of **betweenness** often
nominates a **different** set of "streets to worry about":

- **Betweenness** rewards *through*-routes -> lights up the nearest parallel
  through-corridor when the route is removed.
- **Closeness** rewards being *near everything* -> highlights the geographic
  core, which barely moves when one corridor is removed.

The disagreement is **not a bug**: the two metrics were answering *different
questions all along*. The planner asked betweenness's question ("which
street absorbs the displaced through-traffic?").

In [ ]:
def top_k_changed(before, after, metric, k=10):
    rows = delta_table(before, after, metric)
    rises = [nid for nid, d, *_ in rows[::-1] if d > 0][:k]
    return set(rises)

btw_top = top_k_changed(cent_before, cent_after, "betweenness", 10)
clo_top = top_k_changed(cent_before, cent_after, "closeness", 10)
overlap = btw_top & clo_top
print(f"top-10 betweenness-rise nodes vs top-10 closeness-rise nodes")
print(f"  shared: {len(overlap)} / 10")
print(f"  => the two metrics nominate {'largely the same' if len(overlap) >= 6 else 'DIFFERENT'} streets")
print("\nTeaching payoff: 'different questions, not a contradiction.' Betweenness")
print("answered 'what absorbs displaced through-traffic?'; closeness answered")
print("'where is the accessibility centre, and did removing the route move it?'")

## 6. Extension (c) — wrong-class: when is a topology-only answer misleading?

*(Meta-extend; the explicit bridge to U3/U4. Mostly reasoning, not code.)*

The whole analysis used **structure only** — no traffic counts, no
origin/destination demand, no time of day. So a topology-only answer
misleads whenever **structure and realized flow diverge**, e.g.:

- a street that is **structurally central but realistically empty** (a
  high-betweenness bypass nobody's actual trips use); or
- a **structurally minor street that is jammed every morning** (a school-run
  or employment-cluster street whose demand topology can't see).

**The fix is a different *class* of model.** Realized flow needs **demand /
OD data and time-of-day**, which topology omits
[@kazerani2009betweenness; @gao2013rethinking]. Where that arrives in the
course: **U3** (real sensor flow, METR-LA) gives measured speeds/volumes;
**U4** (time-varying edge weights) lets this same graph carry the demand- and
time-dependence static betweenness cannot. The point is not to *do* U3/U4
work now — it is to correctly diagnose that static topology is the wrong
*class* of model for the realized-flow question, and name the right class.

In [ ]:
# Illustration: the highest-betweenness street is a STRUCTURAL claim only.
# We have no flow data here -- that's exactly the point of extension (c).
top_btw_node = max(cent_after["betweenness"], key=cent_after["betweenness"].get)
print(f"highest-betweenness node after removal: {top_btw_node}")
print(f"  betweenness = {cent_after['betweenness'][top_btw_node]:.5f}")
print("This predicts heavy THROUGH-traffic from structure alone. To confirm it")
print("is actually busy, we'd need measured flow (U3) and time-of-day (U4) --")
print("data this static-topology model does not contain.")

## 7. Notes

- The only genuinely new operation vs. the demo is **edge removal**
  (`remove_route`) — a copy + `remove_edges_from` + re-LCC. Everything else
  reuses the demo's build + centrality idioms, so lean on what the demo taught.
- **This is one path, not the answer.** Two students with different routes,
  different centralities, or different definitions of "critical" can both be
  fully correct. You're graded on the quality of your direct -> interpret ->
  extend loop and your decision log — not on matching this notebook.